# DAMICORE artifacts: violence against women

This notebook is the only database-facing preparation stage for this hypothesis.
It creates the three corpora and the metadata contract consumed by the experiment
notebooks. It does not execute DAMICORE or interpret clusters.

**Scope:** distinct reports with a female victim registered from January 2020 through
June 2026, restricted to the exploratory violence-related source taxonomy.


In [1]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hypotheses.violence_against_women.scripts.create_artifacts import (
    acquire_preparation_lock,
    configured_database_url,
    initialize_preparation,
    prepare_case_artifacts,
    prepare_normalized_artifacts,
    prepare_source_data,
    write_artifact_manifests,
)
from hypotheses.violence_against_women.scripts.experiment_common import CASE_SEEDS

load_dotenv(PROJECT_ROOT / ".env")
CATEGORY_SET_VERSION = os.getenv("DAMICORE_CATEGORY_SET_VERSION", "v2_30")
ARTIFACT_WORKERS = 2
DATABASE_URL = configured_database_url()
PREPARATION_LOCK = acquire_preparation_lock(CATEGORY_SET_VERSION, ARTIFACT_WORKERS)
try:
    PATHS = initialize_preparation(CATEGORY_SET_VERSION, ARTIFACT_WORKERS)
except BaseException:
    PREPARATION_LOCK.release()
    raise


/Users/erickpatrickbarcelos/codes/data-mining/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Query scope and shared context counts

`source_hash` is used only inside PostgreSQL for distinct-report counts. It is not
written to the derived artifacts.


In [2]:
with PREPARATION_LOCK.stage("source"):
    coverage, context_counts, prepared_case_records = prepare_source_data(DATABASE_URL, PATHS)
    display(coverage)
    display(context_counts.head())
    print(f"Context rows: {len(context_counts):,}")
    del coverage


,female_reports,reports_with_category,reports_without_violation
0,1421122,1420747,0


,category,dimension,value,report_count
0,DIREITOS CIVIS E POLÍTICOS > ACESSO À INFORMAÇÃO,ambiente,AMBIENTE DE LAZER/ESPORTE/ENTRETENIMENTO,11
1,DIREITOS CIVIS E POLÍTICOS > ACESSO À INFORMAÇÃO,ambiente,AMBIENTE VIRTUAL (NO ÂMBITO DA INTERNET),139
2,DIREITOS CIVIS E POLÍTICOS > ACESSO À INFORMAÇÃO,ambiente,BERÇÁRIO/CRECHE,7
3,DIREITOS CIVIS E POLÍTICOS > ACESSO À INFORMAÇÃO,ambiente,CASA DA VÍTIMA,1009
4,DIREITOS CIVIS E POLÍTICOS > ACESSO À INFORMAÇÃO,ambiente,CASA DE FAMILIARES,9


Context rows: 839,302


## 2. Select eligible categories and create the normalized corpus

The selection rule follows the source taxonomy and the minimum support threshold used
by the original experiment. Category order is fixed here and reused by every notebook.


In [3]:
with PREPARATION_LOCK.stage("normalized"):
    category_order, included_support, category_map, normalized_corpus_bytes = (
        prepare_normalized_artifacts(context_counts, PATHS)
    )
    del context_counts
    display(category_map)
    print(f"Included categories: {len(category_order)}")
    print(f"Normalized corpus bytes per category: {normalized_corpus_bytes:,}")


,label,category,support
0,category-001.txt,LIBERDADE > SEXUAL > FÍSICA > ESTUPRO,67843
1,category-002.txt,LIBERDADE > SEXUAL > ESTUPRO DE VULNERÁVEL,43494
2,category-003.txt,LIBERDADE > SEXUAL > ESTUPRO DE VULNERÁVEL > P...,14197
3,category-004.txt,LIBERDADE > SEXUAL > ESTUPRO VIRTUAL,2393
4,category-005.txt,LIBERDADE > SEXUAL > ESTUPRO CORRETIVO,399
5,category-006.txt,LIBERDADE > SEXUAL > FÍSICA > ABUSO SEXUAL FÍSICO,28137
6,category-007.txt,LIBERDADE > SEXUAL > FÍSICA > EXPLORAÇÃO SEXUAL,19920
7,category-008.txt,LIBERDADE > SEXUAL > IMPORTUNAÇÃO SEXUAL,27808
8,category-009.txt,LIBERDADE > SEXUAL > PSÍQUICA > ABUSO SEXUAL P...,42555
9,category-010.txt,LIBERDADE > SEXUAL > PSÍQUICA > ASSÉDIO SEXUAL,32323


Included categories: 30
Normalized corpus bytes per category: 23,938


## 3. Create the case-full and case-balanced corpora

The case grain is `source_hash + category` in memory. Canonical case documents contain
only the 20 contextual dimensions, so report identifiers are not exported.


In [4]:
with PREPARATION_LOCK.stage("cases"):
    case_category_map, balanced_sample_size, case_count = prepare_case_artifacts(
        prepared_case_records,
        category_order,
        included_support,
        category_map,
        PATHS,
        workers=ARTIFACT_WORKERS,
    )
    del prepared_case_records, included_support, category_map
    print(f"Case records in memory: {case_count:,}")
    print(f"Balanced sample size per category and replica: {balanced_sample_size:,}")
    print(f"Balanced replicas: {len(CASE_SEEDS)}")
    display(case_category_map.head())


Case records in memory: 3,295,105
Balanced sample size per category and replica: 98
Balanced replicas: 5


,label,category,regime,replicate,case_count,bytes,sha256
0,category-030.txt,DIREITOS CIVIS E POLÍTICOS > VIOLÊNCIA POLITÍC...,case-full,full,247,183957,8fb2de4f26da5f56585c647ee406ebbf02120ba0291e34...
1,category-012.txt,INTEGRIDADE > FÍSICA > AGRESSÃO ou VIAS DE FATO,case-full,full,323166,241060270,c6e29385f7da4940b2036d7164583183411726e49d1953...
2,category-013.txt,INTEGRIDADE > FÍSICA > LESÃO CORPORAL,case-full,full,121922,90557258,94c7ca21347eb8485d22f1f8164ebe45faf39c4de62be7...
3,category-014.txt,INTEGRIDADE > FÍSICA > MAUS TRATOS,case-full,full,651748,482844896,187b41383b92f3fbce6181bdd2ed2f8eb544458909c0be...
4,category-015.txt,INTEGRIDADE > FÍSICA > TORTURA FÍSICA,case-full,full,56421,41821292,7627a9e4dda8a05e5f915101988c7bb8b288189b3b2a23...


## 4. Write the artifact manifest

The manifest is the compatibility contract for the three experiment notebooks.


In [5]:
with PREPARATION_LOCK.stage("manifest"):
    manifest = write_artifact_manifests(PATHS, category_order, balanced_sample_size)
    print("Artifact manifest written.")


Artifact manifest written.
